In [85]:
import torch
import pandas as pd 
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, dataset
from torchvision import transforms
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import Counter
from itertools import product
import re

In [86]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [87]:
def tokenize(text):
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^\w\s]", "", text.lower())
    return text.split()

In [88]:
def build_vocab(texts, max_words=10000):
    counter = Counter()
    
    for text in texts:
        counter.update(tokenize(text))
        
    most_common = counter.most_common(max_words - 2)
   
    vocab = {"<PAD>":0, "<UNK>":1}
    
    for word, _ in most_common:
        vocab[word] = len(vocab)
    
    return vocab


In [89]:
class IMDBDataset(dataset.Dataset):
    def __init__(self, df, vocab, max_len=200):
        self.labels = [1 if label == "positive" else 0 for label in df["sentiment"].values]
        self.max_len = max_len
        self.vocab = vocab
        self.encoded_reviews = []

        # 2. Yorumları Sayısal ID Dizisine Çevirme ve Pre-Padding:
        for review in df["review"].values:
            tokens = tokenize(review)
            ids = [self.vocab.get(token, 1) for token in tokens]

            if len(ids) < self.max_len:
                ids = [0] * (self.max_len - len(ids)) + ids
            else:
                ids = ids[:self.max_len]

            self.encoded_reviews.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.encoded_reviews[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.float)

In [90]:
df = pd.read_csv("C:/Users/Emirhan/Desktop/Pytorch ile derin öğrenme/Rnn/data/IMDB Dataset.csv")

train_df = df.sample(frac=0.8, random_state=42)
test_df = df.drop(train_df.index).reset_index(drop=True)
train_df = train_df.reset_index(drop=True)

In [91]:
vocab = build_vocab(train_df["review"].values, max_words=10000)

train_df = IMDBDataset(train_df, vocab, max_len=200)
test_df = IMDBDataset(test_df, vocab, max_len=200)

train_loader = DataLoader(train_df, batch_size=64, shuffle=True)
test_loader = DataLoader(test_df, batch_size=64, shuffle=False)

sample_x, sample_y = next(iter(train_loader))
sample_x
sample_y

tensor([1., 1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 1.,
        0., 1., 0., 1., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0.,
        1., 1., 0., 1., 1., 0., 1., 0., 1., 0., 0., 1., 1., 0., 1., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 1., 0., 1., 0.])

In [92]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers=1):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, (hn,cn) = self.lstm(x)
        x = self.fc(hn[-1])
        return x

In [93]:
def train_model(model, train_loader, optimizer, criterion, num_epochs = 10, learning_rate = 0.001):
    model.train()
    train_losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(inputs)
            loss = criterion(output.squeeze(1), labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")
    return train_losses
    

In [94]:
def evaluate_model(model, test_loader, criterion):
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for input, label in test_loader:
            input, label = input.to(device), label.to(device)
            output = model(input)
            loss = criterion(output.squeeze(1), label)
            test_loss += loss.item()
            predicted = torch.round(torch.sigmoid(output.squeeze(1)))
            total += label.size(0)
            correct += (predicted == label).sum().item()

    avg_loss = test_loss / len(test_loader)
    accuracy = correct / total
    print(f"Test Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4f}")
    return avg_loss, accuracy

In [95]:
if __name__ == "__main__":
    model = LSTMModel(vocab_size=len(vocab), embedding_dim=100, hidden_dim=128, num_layers=1).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    train_model(model, train_loader, optimizer, criterion, num_epochs=10)
    evaluate_model(model, test_loader, criterion)

Epoch [1/10], Loss: 0.6298
Epoch [2/10], Loss: 0.5999
Epoch [3/10], Loss: 0.5330
Epoch [4/10], Loss: 0.3844
Epoch [5/10], Loss: 0.3012
Epoch [6/10], Loss: 0.2486
Epoch [7/10], Loss: 0.2128
Epoch [8/10], Loss: 0.1871
Epoch [9/10], Loss: 0.1563
Epoch [10/10], Loss: 0.1227
Test Loss: 0.3692, Accuracy: 0.8621
